# DantinoX — Paradigm Profiling Notebook
**AR vs Discrete Diffusion (LLaDA) vs Continuous Diffusion (ELF)**

Builds models from scratch using only `dantinox.core.*` and profiles them
with `dantinox.profiling.*`. All figures — 2D matplotlib and interactive 3D Plotly — are
generated through DantinoX plotting APIs with no manual figure code.

**What it produces:**
- 2-D figures (paradigm-bench style): scale, batch, steps, dtype, composite panel
- Interactive 3-D Plotly surfaces: throughput, latency, steps — rotate with mouse

> **GPU required.** Run → Runtime → Change runtime type → GPU (T4 or better).
>
> **Quick mode** (`QUICK = True`) finishes in ~10 min on T4; set `False` for full sweep (~40 min).

In [ ]:
import subprocess, sys
def _run(cmd): subprocess.run(cmd, shell=True, check=True, capture_output=True)

_run('pip install -q git+https://github.com/your-org/DantinoX.git')
_run('pip install -q plotly pandas')
print('packages installed')

In [ ]:
QUICK = True   # set False for full sweep (larger sizes, more batch points)

import math, warnings
import jax, jax.numpy as jnp
import numpy as np
import pandas as pd
from flax import nnx

from dantinox.core.config import ModelConfig, ELFConfig
from dantinox.core.model   import Transformer
from dantinox.core.elf     import ELFTransformer

from dantinox.profiling import (
    LatencyMetric, FLOPsMetric,
    ThroughputResult, RunProfile, MultiRunReport,
    # 3D interactive (Plotly, rotatable)
    plot_3d_from_df,
    plot_3d_surface, plot_3d_compare,
    # 2D benchmark figures (matplotlib)
    plot_scale, plot_batch, plot_steps, plot_dtype, plot_composite,
    # scalar bar chart
    plot_bar_compare,
)

warnings.filterwarnings('ignore')
jax.config.update('jax_threefry_partitionable', True)
print(f'JAX {jax.__version__}  backend={jax.default_backend()}')
print(f'device: {jax.devices()[0].device_kind}')

## 1 — Model factory
Shared backbone sizes across all three paradigms. Same dim/heads/blocks so differences reflect the *paradigm*, not the architecture.

In [ ]:
# (dim, n_heads, head_size, num_blocks)
SIZES = {
    'tiny':   (64,  4,  16, 2),
    'small':  (128, 4,  32, 3),
    'medium': (256, 8,  32, 6),
    'large':  (512, 16, 32, 8),
    'xl':     (512, 16, 32, 16),
}
SIZES_QUICK = ['tiny', 'small', 'medium', 'large']
VOCAB   = 256
MASK_ID = 4

def _cfg_ar(size, seq_len=256):
    dim, nh, hs, nb = SIZES[size]
    return ModelConfig(dim=dim, n_heads=nh, head_size=hs, num_blocks=nb,
                       vocab_size=VOCAB, max_context=seq_len + 1,
                       causal=True, dropout=0.0)

def _cfg_disc(size, seq_len=256):
    dim, nh, hs, nb = SIZES[size]
    return ModelConfig(dim=dim, n_heads=nh, head_size=hs, num_blocks=nb,
                       vocab_size=VOCAB, max_context=seq_len + 1,
                       causal=False, dropout=0.0, mask_token_id=MASK_ID)

def _cfg_elf(size, seq_len=256):
    dim, nh, hs, nb = SIZES[size]
    return ELFConfig(embed_dim=dim, bottleneck_dim=max(32, dim // 2),
                     model_dim=dim, n_heads=nh, head_size=hs, num_blocks=nb,
                     vocab_size=VOCAB, max_seq_len=seq_len,
                     gradient_checkpointing=False, dropout=0.0)

def _params_m(model):
    _, st = nnx.split(model)
    return sum(x.size for x in jax.tree_util.tree_leaves(st) if hasattr(x, 'size')) / 1e6

def _cast_bf16(model):
    params = nnx.state(model, nnx.Param)
    nnx.update(model, jax.tree_util.tree_map(
        lambda x: x.astype(jnp.bfloat16) if jnp.issubdtype(x.dtype, jnp.floating) else x,
        params))

print('Model factory ready.')

## 2 — Generation closures
Each closure wraps a full generation loop. Passed to `LatencyMetric.measure(fn, n_tokens=B×G)`.

In [ ]:
@nnx.jit
def _ar_prefill(model, x):
    return model(x, caches=None, cache_index=0, deterministic=True).kv_caches

@nnx.jit
def _ar_decode(model, tok, cache, pos):
    out = model(tok, caches=cache, cache_index=pos, deterministic=True)
    return jnp.argmax(out.logits[:, -1, :], -1)[:, None].astype(jnp.int32), out.kv_caches

@nnx.jit
def _disc_step(model, x_t):
    out = model(x_t, deterministic=True)
    return jnp.where(x_t == MASK_ID, jnp.argmax(out.logits, -1).astype(jnp.int32), x_t)

@nnx.jit
def _elf_step(model, z, t_arr, dt):
    x_prev = jnp.zeros_like(z)
    w      = jnp.ones(z.shape[0], dtype=z.dtype)
    mask   = jnp.zeros(z.shape[0], dtype=bool)
    out    = model(z, x_prev, t_arr, w, mask, deterministic=True)
    v      = (out.x_pred - z) / jnp.clip(1.0 - t_arr[:, None, None], 1e-6)
    return z + dt * v


def make_ar_gen(model, B, P, G):
    prompt = jnp.ones((B, P), jnp.int32)
    tok0   = jnp.ones((B, 1), jnp.int32)
    def fn():
        cache = _ar_prefill(model, prompt)
        tok   = tok0
        for i in range(G):
            tok, cache = _ar_decode(model, tok, cache, jnp.array(P + i, jnp.int32))
        jax.block_until_ready(tok)
    return fn

def make_disc_gen(model, B, G, steps):
    x_init = jnp.full((B, G), MASK_ID, jnp.int32)
    def fn():
        x = x_init
        for _ in range(steps):
            x = _disc_step(model, x)
        jax.block_until_ready(x)
    return fn

def make_elf_gen(model, B, G, steps, dim):
    z    = jax.random.normal(jax.random.key(0), (B, G, dim), jnp.float32)
    dt   = jnp.array(1.0 / steps, jnp.float32)
    ts   = jnp.linspace(0.0, 1.0 - 1.0 / steps, steps)
    def fn():
        zz = z
        for i in range(steps):
            zz = _elf_step(model, zz, jnp.full((B,), ts[i], jnp.float32), dt)
        jax.block_until_ready(zz)
    return fn

print('Generation closures ready.')

## 3 — Scale sweep
Profiles each (paradigm × size) and collects `params_m`, `tok_s`, `step_ms`, `mfu_pct`.

In [ ]:
PEAK_TFLOPS   = 312.0   # bf16 A100; T4 = 65 TFLOPS
N_WARMUP      = 1 if QUICK else 3
N_MEASURE     = 3 if QUICK else 10
B_DEFAULT     = 4
G_DEFAULT     = 128
P_DEFAULT     = 64
STEPS_DEFAULT = 32

sizes_to_run = SIZES_QUICK if QUICK else list(SIZES.keys())
lat = LatencyMetric(n_warmup=N_WARMUP, n_measure=N_MEASURE)

scale_rows = []
for size in sizes_to_run:
    dim, n_heads, head_size, num_blocks = SIZES[size]
    print(f'  [{size}]', end=' ', flush=True)
    for paradigm in ('AR', 'Discrete', 'Continuous'):
        try:
            if paradigm == 'AR':
                model = Transformer(_cfg_ar(size, seq_len=P_DEFAULT+G_DEFAULT), rngs=nnx.Rngs(0))
                fn    = make_ar_gen(model, B_DEFAULT, P_DEFAULT, G_DEFAULT)
                tok   = jnp.ones((B_DEFAULT, 1), jnp.int32)
                cache = _ar_prefill(model, jnp.ones((B_DEFAULT, P_DEFAULT), jnp.int32))
                step_fn = lambda: _ar_decode(model, tok, cache, jnp.array(P_DEFAULT, jnp.int32))
            elif paradigm == 'Discrete':
                model = Transformer(_cfg_disc(size, seq_len=G_DEFAULT), rngs=nnx.Rngs(0))
                fn    = make_disc_gen(model, B_DEFAULT, G_DEFAULT, STEPS_DEFAULT)
                x     = jnp.full((B_DEFAULT, G_DEFAULT), MASK_ID, jnp.int32)
                step_fn = lambda: _disc_step(model, x)
            else:
                model = ELFTransformer(_cfg_elf(size, seq_len=G_DEFAULT), rngs=nnx.Rngs(0))
                fn    = make_elf_gen(model, B_DEFAULT, G_DEFAULT, STEPS_DEFAULT, dim)
                z_s   = jax.random.normal(jax.random.key(0), (B_DEFAULT, G_DEFAULT, dim))
                t_s   = jnp.full((B_DEFAULT,), 0.5, jnp.float32)
                dt_s  = jnp.array(1.0 / STEPS_DEFAULT, jnp.float32)
                step_fn = lambda: _elf_step(model, z_s, t_s, dt_s)

            r_step = lat.measure(step_fn, n_tokens=B_DEFAULT * G_DEFAULT)
            r_e2e  = lat.measure(fn,      n_tokens=B_DEFAULT * G_DEFAULT)
            tok_s  = B_DEFAULT * G_DEFAULT * 1e3 / r_e2e.mean_ms

            # analytical MFU
            ffn_h   = 4 * dim
            step_gf = num_blocks * (
                2*B_DEFAULT*G_DEFAULT*dim*dim
                + 4*B_DEFAULT*G_DEFAULT*dim*(n_heads*head_size)
                + 4*B_DEFAULT*n_heads*G_DEFAULT*G_DEFAULT*head_size
                + 2*B_DEFAULT*G_DEFAULT*dim*dim
                + 4*B_DEFAULT*G_DEFAULT*dim*ffn_h
            ) / 1e9
            n_steps = G_DEFAULT if paradigm == 'AR' else STEPS_DEFAULT
            mfu = (100 * step_gf * n_steps * 1e9
                   / (r_e2e.mean_ms / 1e3) / (PEAK_TFLOPS * 1e12))

            scale_rows.append(dict(paradigm=paradigm, size=size, params_m=_params_m(model),
                                   e2e_ms=r_e2e.mean_ms, step_ms=r_step.mean_ms,
                                   tok_s=tok_s, mfu_pct=mfu))
            print(f'{paradigm}\u2713', end=' ', flush=True)
        except Exception as e:
            print(f'{paradigm}\u2717({e.__class__.__name__})', end=' ', flush=True)
    print()

scale_df = pd.DataFrame(scale_rows)
print(f'Scale sweep done — {len(scale_df)} rows')
scale_df.round(2)

## 4 — Batch-size × seq_len sweep
2-D grid: `batch_size × seq_len → tps`. Used for 2-D and 3-D throughput plots.

In [ ]:
BATCH_SIZES = [1, 2, 4, 8, 16, 32]  if QUICK else [1, 2, 4, 8, 16, 32, 64, 128]
SEQ_LENS    = [64, 128, 256]         if QUICK else [32, 64, 128, 256, 512]
BENCH_SIZE  = 'medium'

batch_rows = []
throughput_results = {}   # kept for plot_bar_compare

for paradigm in ('AR', 'Discrete', 'Continuous'):
    print(f'  {paradigm}\u2026', end=' ', flush=True)
    dim, *_ = SIZES[BENCH_SIZE]
    try:
        if paradigm == 'AR':
            model = Transformer(_cfg_ar(BENCH_SIZE, seq_len=P_DEFAULT+max(SEQ_LENS)), rngs=nnx.Rngs(0))
            make_fn = lambda bs, sl, m=model: make_ar_gen(m, bs, P_DEFAULT, sl)
        elif paradigm == 'Discrete':
            model = Transformer(_cfg_disc(BENCH_SIZE, seq_len=max(SEQ_LENS)), rngs=nnx.Rngs(0))
            make_fn = lambda bs, sl, m=model: make_disc_gen(m, bs, sl, STEPS_DEFAULT)
        else:
            model = ELFTransformer(_cfg_elf(BENCH_SIZE, seq_len=max(SEQ_LENS)), rngs=nnx.Rngs(0))
            make_fn = lambda bs, sl, m=model: make_elf_gen(m, bs, sl, STEPS_DEFAULT, dim)

        grid, by_batch, by_seq = [], {}, {}
        for bs in BATCH_SIZES:
            for sl in SEQ_LENS:
                try:
                    r   = lat.measure(make_fn(bs, sl), n_tokens=bs * sl)
                    tps = bs * sl * 1e3 / r.mean_ms
                    grid.append({'batch_size': bs, 'seq_len': sl, 'tps': tps})
                    if sl == SEQ_LENS[1]: by_batch[bs] = tps
                    if bs == 1:           by_seq[sl]   = tps
                    batch_rows.append(dict(paradigm=paradigm, batch_size=bs, seq_len=sl,
                                          tps=tps, e2e_ms=r.mean_ms))
                    print('.', end='', flush=True)
                except Exception:
                    break

        all_tps = [e['tps'] for e in grid]
        throughput_results[paradigm] = ThroughputResult(
            peak_tps=max(all_tps) if all_tps else float('nan'),
            by_batch=by_batch, by_seq=by_seq, seq_len=SEQ_LENS[1], grid=grid,
        )
        print(f'  {len(grid)} pts, peak={max(all_tps)/1e3:.1f}k tok/s')
    except Exception as e:
        print(f'FAILED: {e}')

batch_df = pd.DataFrame(batch_rows)
print('Batch sweep done.')

## 5 — Diffusion-steps sweep
AR is plotted as a flat reference line.

In [ ]:
STEPS_LIST = [4, 8, 16, 32] if QUICK else [4, 8, 16, 32, 64, 128]

steps_rows = []
dim, *_ = SIZES[BENCH_SIZE]

for paradigm in ('AR', 'Discrete', 'Continuous'):
    print(f'  {paradigm}\u2026', end=' ', flush=True)
    try:
        if paradigm == 'AR':
            model = Transformer(_cfg_ar(BENCH_SIZE, seq_len=P_DEFAULT+G_DEFAULT), rngs=nnx.Rngs(0))
            r   = lat.measure(make_ar_gen(model, B_DEFAULT, P_DEFAULT, G_DEFAULT),
                              n_tokens=B_DEFAULT * G_DEFAULT)
            tps = B_DEFAULT * G_DEFAULT * 1e3 / r.mean_ms
            steps_rows += [dict(paradigm=paradigm, steps=s, tps=tps, e2e_ms=r.mean_ms)
                           for s in STEPS_LIST]
        elif paradigm == 'Discrete':
            model = Transformer(_cfg_disc(BENCH_SIZE, seq_len=G_DEFAULT), rngs=nnx.Rngs(0))
            for s in STEPS_LIST:
                r   = lat.measure(make_disc_gen(model, B_DEFAULT, G_DEFAULT, s),
                                  n_tokens=B_DEFAULT * G_DEFAULT)
                tps = B_DEFAULT * G_DEFAULT * 1e3 / r.mean_ms
                steps_rows.append(dict(paradigm=paradigm, steps=s, tps=tps, e2e_ms=r.mean_ms))
                print(f'S={s}\u2713', end=' ', flush=True)
        else:
            model = ELFTransformer(_cfg_elf(BENCH_SIZE, seq_len=G_DEFAULT), rngs=nnx.Rngs(0))
            for s in STEPS_LIST:
                r   = lat.measure(make_elf_gen(model, B_DEFAULT, G_DEFAULT, s, dim),
                                  n_tokens=B_DEFAULT * G_DEFAULT)
                tps = B_DEFAULT * G_DEFAULT * 1e3 / r.mean_ms
                steps_rows.append(dict(paradigm=paradigm, steps=s, tps=tps, e2e_ms=r.mean_ms))
                print(f'S={s}\u2713', end=' ', flush=True)
        print()
    except Exception as e:
        print(f'FAILED: {e}')

steps_df = pd.DataFrame(steps_rows)
print('Steps sweep done.')

## 6 — dtype sweep (fp32 vs bf16)

In [ ]:
dtype_rows = []
dim, *_ = SIZES[BENCH_SIZE]

for paradigm in ('AR', 'Discrete', 'Continuous'):
    for bf16 in (False, True):
        label = 'bf16' if bf16 else 'fp32'
        try:
            if paradigm == 'AR':
                model = Transformer(_cfg_ar(BENCH_SIZE, seq_len=P_DEFAULT+G_DEFAULT), rngs=nnx.Rngs(0))
                if bf16: _cast_bf16(model)
                fn = make_ar_gen(model, B_DEFAULT, P_DEFAULT, G_DEFAULT)
            elif paradigm == 'Discrete':
                model = Transformer(_cfg_disc(BENCH_SIZE, seq_len=G_DEFAULT), rngs=nnx.Rngs(0))
                if bf16: _cast_bf16(model)
                fn = make_disc_gen(model, B_DEFAULT, G_DEFAULT, STEPS_DEFAULT)
            else:
                model = ELFTransformer(_cfg_elf(BENCH_SIZE, seq_len=G_DEFAULT), rngs=nnx.Rngs(0))
                if bf16: _cast_bf16(model)
                fn = make_elf_gen(model, B_DEFAULT, G_DEFAULT, STEPS_DEFAULT, dim)

            r   = lat.measure(fn, n_tokens=B_DEFAULT * G_DEFAULT)
            tps = B_DEFAULT * G_DEFAULT * 1e3 / r.mean_ms
            dtype_rows.append(dict(paradigm=paradigm, dtype=label, tps=tps, e2e_ms=r.mean_ms))
            print(f'  {paradigm}/{label}: {tps:.0f} tok/s')
        except Exception as e:
            print(f'  {paradigm}/{label}: FAILED {e}')

dtype_df = pd.DataFrame(dtype_rows)
print('dtype sweep done.')

## 7 — 2-D figures via DantinoX plot APIs

All five benchmark figures are generated with a single `dantinox.profiling` call each.
Return value is `(fig, axes)` so you can further customise any panel.

In [ ]:
# Fig 1 — throughput vs model size + speedup ratio
fig, axes = plot_scale(scale_df)

In [ ]:
# Fig 2 — throughput vs batch_size (two panels: first and last seq_len)
fig, axes = plot_batch(batch_df, seq_lens=[SEQ_LENS[0], SEQ_LENS[-1]])

In [ ]:
# Fig 3 — throughput + latency vs diffusion steps
fig, axes = plot_steps(steps_df)

In [ ]:
# Fig 4 — fp32 vs bf16
fig, ax = plot_dtype(dtype_df)

In [ ]:
# Fig 5 — 2x2 composite panel (MFU, step latency, Pareto frontier, steps scatter)
fig, axes = plot_composite(scale_df, batch_df, steps_df)

## 8 — Interactive 3-D Plotly surfaces

`plot_3d_from_df` renders a rotatable 3-D surface or scatter directly from any sweep DataFrame —
no `RunProfile` or `MultiRunReport` boilerplate needed.

**Rotate** by clicking and dragging · **Zoom** with scroll · **Hover** for exact values.

In [ ]:
# 3D-1: single-paradigm surface (Discrete Diffusion)
fig = plot_3d_from_df(
    batch_df[batch_df.paradigm == 'Discrete'],
    z='tps', title='Throughput surface — Masked Diffusion (LLaDA)',
)
fig.show()

In [ ]:
# 3D-2: three-paradigm overlay (surface mode — rotate to compare heights)
fig = plot_3d_from_df(
    batch_df, z='tps', mode='surface',
    title='Throughput (tok/s): AR vs Discrete vs ELF — rotate to explore!',
)
fig.show()

In [ ]:
# 3D-3: scatter mode (each dot = one measurement point)
fig = plot_3d_from_df(
    batch_df, z='tps', mode='scatter',
    title='Throughput scatter: each point = (batch_size, seq_len, tok/s)',
)
fig.show()

In [ ]:
# Latency sweep → lat_df  (batch_size × seq_len → mean_ms / p50 / p95 / p99)
lat_rows = []
dim, *_ = SIZES[BENCH_SIZE]

for paradigm in ('AR', 'Discrete', 'Continuous'):
    print(f'  {paradigm}\u2026', end=' ', flush=True)
    try:
        if paradigm == 'AR':
            model = Transformer(_cfg_ar(BENCH_SIZE, seq_len=P_DEFAULT+max(SEQ_LENS)), rngs=nnx.Rngs(0))
            make_fn = lambda bs, sl, m=model: make_ar_gen(m, bs, P_DEFAULT, sl)
        elif paradigm == 'Discrete':
            model = Transformer(_cfg_disc(BENCH_SIZE, seq_len=max(SEQ_LENS)), rngs=nnx.Rngs(0))
            make_fn = lambda bs, sl, m=model: make_disc_gen(m, bs, sl, STEPS_DEFAULT)
        else:
            model = ELFTransformer(_cfg_elf(BENCH_SIZE, seq_len=max(SEQ_LENS)), rngs=nnx.Rngs(0))
            make_fn = lambda bs, sl, m=model: make_elf_gen(m, bs, sl, STEPS_DEFAULT, dim)

        for bs in BATCH_SIZES[:4]:
            for sl in SEQ_LENS:
                try:
                    r = lat.measure(make_fn(bs, sl), n_tokens=bs * sl)
                    lat_rows.append(dict(paradigm=paradigm, batch_size=bs, seq_len=sl,
                                         mean_ms=r.mean_ms, p50_ms=r.p50_ms,
                                         p95_ms=r.p95_ms, p99_ms=r.p99_ms))
                    print('.', end='', flush=True)
                except Exception:
                    break
        print()
    except Exception as e:
        print(f'FAILED: {e}')

lat_df = pd.DataFrame(lat_rows)
print(f'Latency sweep done — {len(lat_df)} points')

In [ ]:
# 3D-4: mean latency surface
fig = plot_3d_from_df(
    lat_df, z='mean_ms', mode='surface',
    title='E2E latency (ms): batch_size \u00d7 seq_len',
)
fig.show()

In [ ]:
# 3D-5: p99 tail latency scatter
fig = plot_3d_from_df(
    lat_df, z='p99_ms', mode='scatter',
    title='p99 latency (ms) — tail latency landscape',
)
fig.show()

In [ ]:
# Steps-grid sweep → steps_grid_df  (batch_size × n_steps → tps)
steps_grid_rows = []
dim, *_ = SIZES[BENCH_SIZE]

for paradigm in ('Discrete', 'Continuous'):
    print(f'  {paradigm}\u2026', end=' ', flush=True)
    try:
        if paradigm == 'Discrete':
            model = Transformer(_cfg_disc(BENCH_SIZE, seq_len=G_DEFAULT), rngs=nnx.Rngs(0))
            make_fn = lambda bs, s, m=model: make_disc_gen(m, bs, G_DEFAULT, s)
        else:
            model = ELFTransformer(_cfg_elf(BENCH_SIZE, seq_len=G_DEFAULT), rngs=nnx.Rngs(0))
            make_fn = lambda bs, s, m=model: make_elf_gen(m, bs, G_DEFAULT, s, dim)

        for bs in BATCH_SIZES[:4]:
            for steps in STEPS_LIST:
                try:
                    r   = lat.measure(make_fn(bs, steps), n_tokens=bs * G_DEFAULT)
                    tps = bs * G_DEFAULT * 1e3 / r.mean_ms
                    steps_grid_rows.append(dict(paradigm=paradigm, batch_size=bs,
                                               steps=steps, tps=tps))
                    print('.', end='', flush=True)
                except Exception:
                    break
        print()
    except Exception as e:
        print(f'FAILED: {e}')

steps_grid_df = pd.DataFrame(steps_grid_rows)
print(f'Steps-grid done — {len(steps_grid_df)} points')

In [ ]:
# 3D-6: throughput vs (batch_size, n_steps)
fig = plot_3d_from_df(
    steps_grid_df, z='tps', y_col='steps', mode='surface',
    title='Throughput (tok/s): batch_size \u00d7 n_steps — Discrete vs ELF',
)
fig.update_layout(scene=dict(yaxis=dict(title='n_steps')))
fig.show()

In [ ]:
# Peak throughput bar chart per paradigm
profiles = [
    RunProfile(run_name=p, run_dir=f'/tmp/{p}', config={}, throughput=throughput_results[p])
    for p in ('AR', 'Discrete', 'Continuous') if p in throughput_results
]
report = MultiRunReport(profiles=profiles, total_time_s=0., metrics=['throughput'], filter_used={})
fig = plot_bar_compare(report, metric='peak_tps',
                        title='Peak throughput per paradigm (tok/s, medium backbone)')
fig.show()

## 9 — Save all 3-D figures to HTML
Each file is a standalone interactive HTML you can open locally or share.

In [ ]:
import os
os.makedirs('results/colab_plots', exist_ok=True)

save_list = [
    (plot_3d_from_df(batch_df, z='tps', mode='surface', show=False),          'throughput_3d_surface'),
    (plot_3d_from_df(batch_df, z='tps', mode='scatter', show=False),          'throughput_3d_scatter'),
    (plot_3d_from_df(lat_df, z='mean_ms', mode='surface', show=False),        'latency_3d_surface'),
    (plot_3d_from_df(lat_df, z='p99_ms', mode='scatter', show=False),         'p99_latency_scatter'),
    (plot_3d_from_df(steps_grid_df, z='tps', y_col='steps', show=False),      'throughput_vs_steps_3d'),
]
for fig, name in save_list:
    path = f'results/colab_plots/{name}.html'
    fig.write_html(path, include_plotlyjs='cdn')
    print(f'  saved {path}  ({os.path.getsize(path) // 1024} KB)')

print('Done!')